# BirdCLEF 2026 — Self-Supervised Pretraining + Fine-Tuning (Pipeline 04)

This pipeline implements a **SimCLR-style contrastive learning** approach prior to supervised fine-tuning. It addresses the massive amount of unlabeled or weakly labeled acoustic structure in the data by first learning general acoustic embeddings, improving generalization on rare species.

**Architecture**:
1. **Phase 1 (SSL)**: Unlabeled soundscapes + augmented views → EfficientNet-B0 → Projection Head → NT-Xent Loss
2. **Phase 2 (Fine-Tuning)**: Pretrained Encoder → Classification Head → BCE Multi-label Loss

## 1. Setup & Imports

In [ ]:
import os, gc, sys, math, time, glob, random, ast
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
from pathlib import Path
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
import torchaudio.transforms as T
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

## 2. Configuration & Paths

In [ ]:
class Config:
    ROOT_DIR       = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV      = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    SOUNDSCAPE_CSV = os.path.join(ROOT_DIR, 'train_soundscapes_labels.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    MODEL_DIR = Path('/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth')

    SR         = 32000
    WINDOW_SECONDS = 5

    N_MELS     = 128
    N_FFT      = 2048
    HOP_LENGTH = 512
    FMIN       = 20
    FMAX       = 16000

    # SSL specific config
    SSL_EPOCHS = 10
    SSL_BATCH_SIZE = 64
    TEMPERATURE = 0.1
    PROJECTION_DIM = 128

    # Fine-tuning config
    FT_EPOCHS = 15
    FT_BATCH_SIZE = 32
    FT_LR = 1e-4
    SSL_LR = 5e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0
    MODEL_NAME = 'tf_efficientnet_b0'
    SEED = 42
    NUM_CLASSES = 0

CFG = Config()

def seed_everything(seed):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)

train_df   = pd.read_csv(CFG.TRAIN_CSV)
ss_df      = pd.read_csv(CFG.SOUNDSCAPE_CSV)
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
label_to_id = {label: i for i, label in enumerate(submission_labels)}
train_df['label_id'] = train_df['primary_label'].map(label_to_id)
CFG.NUM_CLASSES = len(submission_labels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 3. Self-Supervised Dataset (Two Views)

In [ ]:
class SSLDataset(Dataset):
    """Returns two differently augmented views of the same audio segment."""
    def __init__(self, df, audio_dir):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def get_audio(self, row):
        path = os.path.join(self.audio_dir, row['filename'])
        try:
            total = sf.info(path).frames
            if total > self.window_samples:
                start = random.randint(0, total - self.window_samples)
                y, _ = sf.read(path, start=start, frames=self.window_samples, always_2d=True)
            else:
                y, _ = sf.read(path, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)
        return y

    def apply_augmentation(self, y):
        if random.random() < 0.5: y = y + 0.005 * np.random.randn(len(y))
        y_t = torch.tensor(y, dtype=torch.float32)
        mel = self.mel_transform(y_t)
        mel = self.amplitude_to_db(mel)
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img = torch.stack([mel, mel, mel])
        # Random spec augment
        if random.random() < 0.8:
            f = random.randint(0, 16)
            f0 = random.randint(0, max(1, CFG.N_MELS - f))
            img[:, f0:f0+f, :] = 0
        if random.random() < 0.8:
            t = random.randint(0, 32)
            t0 = random.randint(0, max(1, img.size(-1) - t))
            img[:, :, t0:t0+t] = 0
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = self.get_audio(row)
        # Create two augmented views of the same audio
        view1 = self.apply_augmentation(y)
        view2 = self.apply_augmentation(y)
        return view1, view2


## 4. Supervised Datasets

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.is_train = is_train
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX)
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])
        try:
            total = sf.info(path).frames
            if total > self.window_samples:
                start = random.randint(0, total - self.window_samples) if self.is_train else 0
                y, _ = sf.read(path, start=start, frames=self.window_samples, always_2d=True)
            else:
                y, _ = sf.read(path, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples: y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)
        if self.is_train and random.random() < 0.5: y = y + 0.005 * np.random.randn(len(y))
        y_t = torch.tensor(y, dtype=torch.float32)
        mel = self.amplitude_to_db(self.mel_transform(y_t))
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img = torch.stack([mel, mel, mel])
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        target[row['label_id']] = 1.0
        if 'secondary_labels' in row and pd.notna(row['secondary_labels']):
            try:
                for sl in ast.literal_eval(row['secondary_labels']):
                    if sl in label_to_id: target[label_to_id[sl]] = 1.0
            except: pass
        return img, target

class SoundscapeDataset(Dataset):
    def __init__(self, df, audio_dir):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        self.mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX)
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])
        h, m, s = map(int, row['start'].split(':'))
        start_s = (h * 3600 + m * 60 + s) * CFG.SR
        try:
            y, _ = sf.read(path, start=start_s, stop=start_s + self.window_samples, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples: y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)
        y_t = torch.tensor(y, dtype=torch.float32)
        mel = self.amplitude_to_db(self.mel_transform(y_t))
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img = torch.stack([mel, mel, mel])
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        for label in str(row['primary_label']).split(';'):
            if label in label_to_id: target[label_to_id[label]] = 1.0
        return img, target


## 5. Model Architecture

In [ ]:
class SSLModel(nn.Module):
    """Encoder + Projection head for Contrastive Learning"""
    def __init__(self, model_name, projection_dim, model_path=None):
        super().__init__()
        if model_path is not None and Path(model_path).exists():
            self.encoder = timm.create_model(model_name, checkpoint_path=model_path, pretrained=False, in_chans=3)
        else:
            self.encoder = timm.create_model(model_name, pretrained=False, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.encoder.classifier.in_features
            self.encoder.classifier = nn.Identity()
        else:
            in_features = self.encoder.get_classifier().in_features
            self.encoder.reset_classifier(0)
            
        self.projector = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, projection_dim)
        )
        
    def forward(self, x):
        feats = self.encoder(x)
        return self.projector(feats), feats

class FineTuneModel(nn.Module):
    """Encoder + Classification head for Fine-Tuning"""
    def __init__(self, encoder, model_name, num_classes):
        super().__init__()
        self.encoder = encoder
        if 'efficientnet' in model_name:
            # Create a dummy to get features size if needed
            dummy = timm.create_model(model_name, pretrained=False)
            in_features = dummy.classifier.in_features
        else:
            in_features = 1280  # approx for B0, dynamic is better
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        feats = self.encoder(x)
        return self.head(feats)


## 6. Training Logic (SSL + Fine-Tune)

In [ ]:
# NT-Xent Loss for SimCLR
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
        self.cosine_similarity = nn.CosineSimilarity(dim=-1)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        representations = torch.cat([z_i, z_j], dim=0)
        similarity_matrix = self.cosine_similarity(representations.unsqueeze(1), representations.unsqueeze(0))
        
        # Ignore self-similarity
        similarity_matrix.fill_diagonal_(-9e15)
        
        # Labels: i is paired with i+batch_size, and i+batch_size is paired with i
        labels = torch.cat([torch.arange(batch_size, 2*batch_size), torch.arange(batch_size)], dim=0).to(z_i.device)
        
        logits = similarity_matrix / self.temperature
        loss = self.criterion(logits, labels)
        return loss

def train_ssl_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0.0
    for v1, v2 in tqdm(loader, desc='SSL Train', leave=False):
        v1, v2 = v1.to(device), v2.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            z1, _ = model(v1)
            z2, _ = model(v2)
            loss = criterion(z1, z2)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def train_ft_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0.0
    for images, targets in tqdm(loader, desc='FT Train', leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def valid_ft_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    preds, true_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Valid', leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        if len(np.unique(true_targets[:, i])) > 1:
            auc_scores.append(roc_auc_score(true_targets[:, i], preds[:, i]))
    final_auc = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_auc


## 7. Main Execution

In [ ]:
df = train_df.copy()
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()
if rare_birds:
    df = pd.concat([df, df[df['label_id'].isin(rare_birds)]], ignore_index=True)
train_df_clips, valid_df_clips = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=CFG.SEED)
train_ss_df, valid_ss_df = train_test_split(ss_df, test_size=0.2, random_state=CFG.SEED)

# ── Phase 1: SSL Pretraining ───────────────────────────────
print('\n>>> PHASE 1: Self-Supervised Pretraining (SimCLR) <<<')
ssl_ds = SSLDataset(train_df_clips, CFG.TRAIN_AUDIO_DIR)
ssl_loader = DataLoader(ssl_ds, batch_size=CFG.SSL_BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)

ssl_model = SSLModel(CFG.MODEL_NAME, CFG.PROJECTION_DIM, CFG.MODEL_DIR).to(device)
ssl_optimizer = optim.AdamW(ssl_model.parameters(), lr=CFG.SSL_LR, weight_decay=CFG.WEIGHT_DECAY)
ssl_criterion = NTXentLoss(temperature=CFG.TEMPERATURE)
scaler = torch.cuda.amp.GradScaler()

for epoch in range(1, CFG.SSL_EPOCHS + 1):
    t0 = time.time()
    loss = train_ssl_epoch(ssl_model, ssl_loader, ssl_optimizer, ssl_criterion, scaler, device)
    print(f'SSL Epoch {epoch}/{CFG.SSL_EPOCHS} | Loss: {loss:.4f} | Time: {int(time.time() - t0)}s')

print('Pretraining complete. Retaining encoder.')
pretrained_encoder = ssl_model.encoder

# ── Phase 2: Supervised Fine-Tuning ─────────────────────────
print('\n>>> PHASE 2: Supervised Fine-Tuning <<<')
def get_sampler(df_subset):
    c = df_subset['label_id'].value_counts().sort_index().values
    w = 1.0 / (c + 1e-6)
    wt = df_subset['label_id'].map(lambda x: w[x] if x < len(w) else 0).values
    return WeightedRandomSampler(weights=wt, num_samples=len(wt), replacement=True)

train_sampler = get_sampler(train_df_clips)
train_ds = ConcatDataset([BirdDataset(train_df_clips, CFG.TRAIN_AUDIO_DIR, is_train=True), SoundscapeDataset(train_ss_df, CFG.SOUNDSCAPE_DIR)])
valid_ds_clips = BirdDataset(valid_df_clips, CFG.TRAIN_AUDIO_DIR, is_train=False)
valid_ds_ss = SoundscapeDataset(valid_ss_df, CFG.SOUNDSCAPE_DIR)

train_loader = DataLoader(train_ds, batch_size=CFG.FT_BATCH_SIZE, sampler=train_sampler, num_workers=CFG.NUM_WORKERS)
valid_loader_clips = DataLoader(valid_ds_clips, batch_size=CFG.FT_BATCH_SIZE, shuffle=False)
valid_loader_ss = DataLoader(valid_ds_ss, batch_size=CFG.FT_BATCH_SIZE, shuffle=False)

ft_model = FineTuneModel(pretrained_encoder, CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
ft_optimizer = optim.AdamW(ft_model.parameters(), lr=CFG.FT_LR, weight_decay=CFG.WEIGHT_DECAY)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=CFG.FT_EPOCHS)
ft_criterion = nn.BCEWithLogitsLoss()

best_score = 0
for epoch in range(1, CFG.FT_EPOCHS + 1):
    t0 = time.time()
    t_loss = train_ft_epoch(ft_model, train_loader, ft_optimizer, ft_criterion, scaler, device)
    _, v_clips = valid_ft_epoch(ft_model, valid_loader_clips, ft_criterion, device)
    _, v_ss = valid_ft_epoch(ft_model, valid_loader_ss, ft_criterion, device)
    ft_scheduler.step()
    print(f'FT Epoch {epoch}/{CFG.FT_EPOCHS} | Loss: {t_loss:.4f} | Val Clips AUC: {v_clips:.4f} | Val SS AUC: {v_ss:.4f}')
    if v_ss > best_score:
        best_score = v_ss
        torch.save(ft_model.state_dict(), 'best_model_ssl_ft.pth')
        print(f'  !!! NEW BEST MODEL SAVED | Val SS AUC: {best_score:.4f} !!!')
